# A5 compact

Step-by-step animation of [`a5compact`](https://github.com/opengeoshub/vgrid/blob/main/vgrid/conversion/dggscompact/a5compact.py) / `a5.core.compact.compact`.

Input: [`a5_16.geojson`](a5_16.geojson) in this folder.

Each pass groups cells by `a5.cell_to_parent`; when all 4 children of a parent are present, they merge into the parent. **1** shows the full input grid; **3a** highlights cyan children and orange **parent** (dashed).

## Install necessary packages

In [ ]:
%pip install vgrid geopandas matplotlib imageio pillow a5
# optional for MP4:
%pip install imageio-ffmpeg

In [ ]:
"""Step-by-step a5 compact animation."""
from collections import defaultdict
from pathlib import Path

import a5
import geopandas as gpd
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon

from vgrid.conversion.dggs2geo.a52geo import a52geo
from vgrid.conversion.dggscompact.a5compact import a5_compact

INPUT_GEOJSON = Path("a5_16.geojson")
A5_ID_FIELD = "a5"
OUT_GIF = "a5_compact.gif"
OUT_MP4 = "a5_compact.mp4"
FRAME_EVERY_MERGE = 16  # a5_grid_5 has many merges; use 1 for every group
CHILDREN_LABEL = "4 children (merge group)"
DPI = 120
SPLIT_ANTIMERIDIAN = False


def cell_patches(cell_polys, facecolor, edgecolor, alpha=0.85, lw=1.0):
    patches = []
    for poly in cell_polys:
        if poly is None or poly.is_empty:
            continue
        patches.append(MplPolygon(list(poly.exterior.coords), closed=True))
    return PatchCollection(
        patches,
        facecolor=facecolor,
        edgecolor=edgecolor,
        alpha=alpha,
        linewidths=lw,
        zorder=2,
    )


def polys_for_ids(cell_ids, split_antimeridian=False):
    polys = []
    for cell_id in cell_ids:
        poly = a52geo(cell_id, split_antimeridian=split_antimeridian)
        if poly is not None and not poly.is_empty:
            polys.append(poly)
    return polys


def render_frame(
    bounds,
    title,
    path,
    background_polys=None,
    child_polys=None,
    parent_poly=None,
):
    fig, ax = plt.subplots(figsize=(8, 8))
    minx, miny, maxx, maxy = bounds
    pad = max(maxx - minx, maxy - miny) * 0.06 or 0.01
    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad)

    if background_polys:
        ax.add_collection(
            cell_patches(background_polys, "#e8eaf6", "#5c6bc0", alpha=0.5, lw=0.8)
        )
    if child_polys:
        ax.add_collection(
            cell_patches(child_polys, "#00bcd4", "#006064", alpha=0.9, lw=1.4)
        )
    if parent_poly is not None:
        ax.add_collection(
            cell_patches([parent_poly], "#ff9800", "#e65100", alpha=0.55, lw=2.0)
        )
        gpd.GeoSeries([parent_poly.boundary]).plot(
            ax=ax, color="#e65100", lw=3.5, linestyle="--", zorder=4
        )

    ax.set_facecolor("#fafafa")
    ax.plot([], [], color="#00bcd4", lw=4, label=CHILDREN_LABEL)
    ax.plot([], [], color="#ff9800", lw=4, label="parent cell")
    ax.plot([], [], color="#5c6bc0", lw=4, label="other cells")
    ax.legend(loc="upper right", fontsize=8, framealpha=0.95)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.grid(False)
    fig.subplots_adjust(left=0.08, right=0.92, top=0.92, bottom=0.08)
    fig.savefig(path, dpi=DPI, facecolor="white")
    plt.close(fig)


def find_merge_groups(current_hexes):
    grouped = defaultdict(set)
    for cell_hex in current_hexes:
        try:
            parent_u64 = a5.cell_to_parent(a5.hex_to_u64(cell_hex))
        except Exception:
            continue
        grouped[parent_u64].add(cell_hex)
    merges = []
    for parent_u64, children_hex in grouped.items():
        all_children_hex = {
            a5.u64_to_hex(u64) for u64 in a5.cell_to_children(parent_u64)
        }
        if children_hex == all_children_hex:
            merges.append((a5.u64_to_hex(parent_u64), sorted(children_hex)))
    return merges


def apply_merges(current_ids, merges):
    new_ids = set(current_ids)
    for parent, children in merges:
        new_ids.difference_update(children)
        new_ids.add(parent)
    return new_ids


def a5_compact_with_frames(a5_ids, bounds, frame_dir, split_antimeridian=False):
    frame_dir.mkdir(parents=True, exist_ok=True)
    frames = []
    idx = 0
    id_to_poly = {}

    def poly_for(cell_id):
        if cell_id not in id_to_poly:
            id_to_poly[cell_id] = a52geo(cell_id, split_antimeridian=split_antimeridian)
        return id_to_poly[cell_id]

    def snap_simple(polys, title):
        nonlocal idx
        p = frame_dir / f"frame_{idx:04d}.png"
        render_frame(bounds, title, p, background_polys=polys)
        frames.append(p)
        idx += 1

    def snap_merge(current_ids, parent_id, child_ids, title):
        nonlocal idx
        child_set = set(child_ids)
        background = [
            poly_for(cid)
            for cid in current_ids
            if cid not in child_set
            and poly_for(cid) is not None
            and not poly_for(cid).is_empty
        ]
        children = [
            poly_for(cid)
            for cid in child_ids
            if poly_for(cid) is not None and not poly_for(cid).is_empty
        ]
        parent_poly = poly_for(parent_id)
        p = frame_dir / f"frame_{idx:04d}.png"
        render_frame(
            bounds,
            title,
            p,
            background_polys=background,
            child_polys=children,
            parent_poly=parent_poly,
        )
        frames.append(p)
        idx += 1

    input_polys = [
        p
        for cid in a5_ids
        for p in [poly_for(cid)]
        if p is not None and not p.is_empty
    ]
    snap_simple(input_polys, f"1. Input grid ({len(a5_ids)} cells)")

    current = set(a5_ids)
    round_idx = 0
    while True:
        merges = find_merge_groups(current)
        if not merges:
            break
        round_idx += 1
        for merge_i, (parent_id, child_ids) in enumerate(merges, start=1):
            if merge_i % FRAME_EVERY_MERGE != 0 and merge_i != len(merges):
                continue
            snap_merge(
                current,
                parent_id,
                child_ids,
                f"3a. Round {round_idx} merge {merge_i}/{len(merges)}: "
                f"{len(child_ids)} children → parent {parent_id}",
            )
        current = apply_merges(current, merges)
        snap_simple(
            polys_for_ids(sorted(current), split_antimeridian),
            f"3b. After round {round_idx}: {len(current)} cells",
        )

    final_ids = sorted(a5_compact(a5_ids))
    snap_simple(
        polys_for_ids(final_ids, split_antimeridian),
        f"4. Final compact set ({len(final_ids)} cells)",
    )
    return frames, final_ids


def main():
    print(f"Using {INPUT_GEOJSON}")
    gdf = gpd.read_file(INPUT_GEOJSON)
    a5_ids = gdf[A5_ID_FIELD].drop_duplicates().tolist()
    if not a5_ids:
        raise ValueError(f"No '{A5_ID_FIELD}' values in {INPUT_GEOJSON}")

    bounds = gdf.total_bounds
    frame_dir = Path("_a5_compact_frames")
    frames, final_ids = a5_compact_with_frames(
        a5_ids, bounds, frame_dir, split_antimeridian=SPLIT_ANTIMERIDIAN
    )
    GIF_FRAME_DURATION = 1.8
    imageio.mimsave(
        OUT_GIF, [imageio.imread(f) for f in frames], duration=GIF_FRAME_DURATION
    )
    print(
        f"Wrote {OUT_GIF} ({len(frames)} frames, "
        f"{len(a5_ids)} → {len(final_ids)} cells)"
    )
    try:
        MP4_FPS = 2.4
        writer = imageio.get_writer(OUT_MP4, fps=MP4_FPS)
        for f in frames:
            writer.append_data(imageio.imread(f))
        writer.close()
        print(f"Wrote {OUT_MP4}")
    except Exception as e:
        print(f"MP4 skipped ({e}). GIF is enough.")


if __name__ == "__main__":
    main()